# Character Relations Agent 테스트 (Production Level)

캐릭터 관계 추출 에이전트 테스트 노트북

## 역할: "Relationship Analyst" (관계 분석가)

### 기본 관계
- **type**: 관계 유형 (ALLY, ENEMY, BETRAYER 등)
- **strength**: 관계 강도 (1-10)

### 심리적 깊이 ✨ NEW
- **public_stance vs private_feeling**: 겉과 속의 분리
- **facade_level**: 가식 레벨 (0=솔직, 10=완전 연기)
- **interaction_style/chemistry**: 케미스트리
- **relationship_origin**: 관계 형성 사건
- **mutual_feelings**: 복합 감정

> **⚠️ 핵심 검증**: 비대칭 관계 + 겉과 속의 분리

In [1]:
import sys, os, json, asyncio
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path: sys.path.insert(0, project_root)
from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))
print(f"Project root: {project_root}")

Project root: c:\jungle\weapon\sto-link-AI-backend


In [2]:
SAMPLE_STORY = """두 사람은 한때 '은빛 여명' 기사단의 동맹이었다. 하지만 5년 전 대전쟁 이후, 카엘은 왕국을 배신하고 '암흑회'에 가담했다. 아린은 아직도 그 이유를 알지 못했다.

'너를 찾고 있었어.' 카엘이 검을 뽑았다. '이번엔 네가 지는 거다.'

아린은 입술을 깨물며 전투 자세를 취했다. '배신자와 할 말은 없어.'"""

def create_base_state(story=SAMPLE_STORY):
    return {"content": story, "completed_agents": [], "errors": [], "messages": []}

def run_async(coro):
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            import nest_asyncio; nest_asyncio.apply()
            return loop.run_until_complete(coro)
        return asyncio.run(coro)
    except: return asyncio.run(coro)

## 1. Relations Agent 실행

In [3]:
from app.agents.extraction.character.relations import relations_extraction_node

async def test_relations():
    print("🔗 Relations Agent 테스트...")
    return await relations_extraction_node(create_base_state())

result = run_async(test_relations())

if result.get('errors'):
    print(f"\n❌ 에러 발생:")
    for err in result.get('errors', []):
        print(f"   {err}")
else:
    relations_data = result.get('char_relations', {})
    print(f"\n✅ 추출 완료:")
    print(f"   - 캐릭터 수: {len(relations_data)}개")
    print(f"   - 이름: {list(relations_data.keys())}")

🔗 Relations Agent 테스트...

✅ 추출 완료:
   - 캐릭터 수: 2개
   - 이름: ['카엘', '아린']


## 2. Human-Readable 출력

In [4]:
relations_data = result.get('char_relations', {})

if not relations_data:
    print("❌ 캐릭터 데이터 없음")
else:
    print("="*70)
    print("🔗 Relations Data (캐릭터별 관계)")
    print("="*70)

    for name, data in relations_data.items():
        print(f"\n🧑 {name}")
        relations = data.get('relations', [])
        if relations:
            for rel in relations:
                target = rel.get('target', 'N/A')
                rel_type = rel.get('type', 'N/A')
                strength = rel.get('strength', 5)
                
                print(f"   → {target}: {rel_type} (강도: {strength}/10)")
        else:
            print("   (관계 없음)")

🔗 Relations Data (캐릭터별 관계)

🧑 카엘
   → 아린: BETRAYER (강도: 8/10)

🧑 아린
   → 카엘: FORMER_ALLY (강도: 7/10)


## 3. 🎭 Public vs Private (겉과 속) - NEW

In [5]:
print("="*70)
print("🎭 Public vs Private (겉과 속의 분리)")
print("="*70)

relations_data = result.get('char_relations', {})

for name, data in relations_data.items():
    print(f"\n🧑 {name}")
    for rel in data.get('relations', []):
        target = rel.get('target', 'N/A')
        public = rel.get('public_stance', 'N/A')
        private = rel.get('private_feeling', 'N/A')
        facade = rel.get('facade_level', 0)
        
        print(f"   → {target}:")
        print(f"      겉 (Public): {public}")
        print(f"      속 (Private): {private}")
        
        # Facade visualization
        facade_bar = '█' * facade + '░' * (10 - facade)
        print(f"      가식 레벨: [{facade_bar}] {facade}/10")
        
        if public != private and public and private:
            print(f"      ⚠️ 겉과 속이 다름!")

🎭 Public vs Private (겉과 속의 분리)

🧑 카엘
   → 아린:
      겉 (Public): ENEMY
      속 (Private): GUILT
      가식 레벨: [██████░░░░] 6/10
      ⚠️ 겉과 속이 다름!

🧑 아린
   → 카엘:
      겉 (Public): ENEMY
      속 (Private): DISTRUST
      가식 레벨: [████░░░░░░] 4/10
      ⚠️ 겉과 속이 다름!


## 4. 🧪 Interaction Style & Chemistry (케미) - NEW

In [6]:
print("="*70)
print("🧪 Interaction Style & Chemistry (케미스트리)")
print("="*70)

relations_data = result.get('char_relations', {})

for name, data in relations_data.items():
    print(f"\n🧑 {name}")
    for rel in data.get('relations', []):
        target = rel.get('target', 'N/A')
        style = rel.get('interaction_style', 'N/A')
        chemistry = rel.get('chemistry', 'N/A')
        
        print(f"   → {target}:")
        print(f"      스타일: {style}")
        print(f"      케미: {chemistry}")

🧪 Interaction Style & Chemistry (케미스트리)

🧑 카엘
   → 아린:
      스타일: Tense_Standoff
      케미: 팽팽한 긴장감

🧑 아린
   → 카엘:
      스타일: Tense_Standoff
      케미: 과거 우정의 아픔


## 5. 🔗 Relationship Origin (관계 기원) - NEW

In [7]:
print("="*70)
print("🔗 Relationship Origin (관계 기원)")
print("="*70)

relations_data = result.get('char_relations', {})

for name, data in relations_data.items():
    print(f"\n🧑 {name}")
    for rel in data.get('relations', []):
        target = rel.get('target', 'N/A')
        origin = rel.get('relationship_origin', {})
        
        if origin and (origin.get('event_description') or origin.get('time_context')):
            print(f"   → {target}:")
            if origin.get('time_context'):
                print(f"      시점: {origin.get('time_context')}")
            if origin.get('event_description'):
                print(f"      사건: {origin.get('event_description')}")
        else:
            print(f"   → {target}: (기원 정보 없음)")

🔗 Relationship Origin (관계 기원)

🧑 카엘
   → 아린:
      시점: 5년 전 대전쟁 이후
      사건: 카엘이 왕국을 배신하고 암흑회에 가담했다

🧑 아린
   → 카엘:
      시점: 5년 전 대전쟁 이후
      사건: 카엘이 왕국을 배신하고 암흑회에 가담했다


## 6. Neo4j Graph 형태

In [8]:
print("="*70)
print("🔗 Neo4j Graph 형태")
print("="*70)

relations_data = result.get('char_relations', {})

edges = []
for name, data in relations_data.items():
    for rel in data.get('relations', []):
        edge = {
            "source": name,
            "target": rel.get('target'),
            "type": rel.get('type'),
            "public": rel.get('public_stance'),
            "private": rel.get('private_feeling')
        }
        edges.append(edge)
        rel_type = rel.get('type', 'RELATED')
        public = rel.get('public_stance', '')
        private = rel.get('private_feeling', '')
        print(f"({name})-[:{rel_type}]->({rel.get('target')})")
        if public != private and public and private:
            print(f"   [겉: {public}, 속: {private}]")

print(f"\nTotal edges: {len(edges)}")

🔗 Neo4j Graph 형태
(카엘)-[:BETRAYER]->(아린)
   [겉: ENEMY, 속: GUILT]
(아린)-[:FORMER_ALLY]->(카엘)
   [겉: ENEMY, 속: DISTRUST]

Total edges: 2


## 7. Full JSON 출력

In [9]:
print("="*70)
print("📄 Full JSON Output")
print("="*70)
relations_data = result.get('char_relations', {})
if relations_data:
    print(json.dumps(relations_data, ensure_ascii=False, indent=2))
else:
    print("{}")

📄 Full JSON Output
{
  "카엘": {
    "name": "카엘",
    "relations": [
      {
        "target": "아린",
        "type": "BETRAYER",
        "strength": 8,
        "description": null,
        "history": "FORMER_ALLY",
        "public_stance": "ENEMY",
        "private_feeling": "GUILT",
        "facade_level": 6,
        "interaction_style": "Tense_Standoff",
        "chemistry": "팽팽한 긴장감",
        "mutual_feelings": [],
        "relationship_origin": {
          "event_ref": null,
          "event_description": "카엘이 왕국을 배신하고 암흑회에 가담했다",
          "time_context": "5년 전 대전쟁 이후"
        }
      }
    ],
    "known_events": [],
    "location_context": null
  },
  "아린": {
    "name": "아린",
    "relations": [
      {
        "target": "카엘",
        "type": "FORMER_ALLY",
        "strength": 7,
        "description": null,
        "history": "ALLY",
        "public_stance": "ENEMY",
        "private_feeling": "DISTRUST",
        "facade_level": 4,
        "interaction_style": "Tense_Standoff",
 

## 8. Production 체크리스트

In [10]:
print("="*70)
print("✅ Production 체크리스트")
print("="*70)

relations_data = result.get('char_relations', {})
checks = []

# 1. 캐릭터 존재
if len(relations_data) >= 2:
    checks.append(("✅", f"{len(relations_data)} characters extracted"))
else:
    checks.append(("❌", "Need at least 2 characters"))

if relations_data:
    # 2. 관계 존재
    total_rels = sum(len(data.get('relations', [])) for data in relations_data.values())
    if total_rels >= 2:
        checks.append(("✅", f"{total_rels} relationships (bilateral)"))
    else:
        checks.append(("⚠️", f"Only {total_rels} relationships"))
    
    # 3. Public vs Private
    has_public_private = any(
        rel.get('public_stance') or rel.get('private_feeling')
        for data in relations_data.values()
        for rel in data.get('relations', [])
    )
    if has_public_private:
        checks.append(("✅", "Public/Private stance extracted"))
    else:
        checks.append(("⚠️", "Missing public/private stance"))
    
    # 4. Interaction Style
    has_style = any(
        rel.get('interaction_style') or rel.get('chemistry')
        for data in relations_data.values()
        for rel in data.get('relations', [])
    )
    if has_style:
        checks.append(("✅", "Interaction style/chemistry extracted"))
    else:
        checks.append(("⚠️", "Missing interaction style"))
    
    # 5. Relationship Origin
    has_origin = any(
        rel.get('relationship_origin', {}).get('event_description')
        for data in relations_data.values()
        for rel in data.get('relations', [])
    )
    if has_origin:
        checks.append(("✅", "Relationship origin extracted"))
    else:
        checks.append(("⚠️", "Missing relationship origin"))
    
    # 6. 비대칭 관계
    arin_rels = relations_data.get('아린', {}).get('relations', [])
    kael_rels = relations_data.get('카엘', {}).get('relations', [])
    arin_to_kael = next((r for r in arin_rels if r.get('target') == '카엘'), None)
    kael_to_arin = next((r for r in kael_rels if r.get('target') == '아린'), None)
    
    if arin_to_kael and kael_to_arin:
        checks.append(("✅", "Bidirectional relationship exists"))
    else:
        checks.append(("❌", "Missing bidirectional relationship"))

print()
for status, msg in checks:
    print(f"{status} {msg}")

print("\n" + "=" * 70)
passed = sum(1 for s, _ in checks if s == "✅")
print(f"결과: {passed}/{len(checks)} checks passed")

✅ Production 체크리스트

✅ 2 characters extracted
✅ 2 relationships (bilateral)
✅ Public/Private stance extracted
✅ Interaction style/chemistry extracted
✅ Relationship origin extracted
✅ Bidirectional relationship exists

결과: 6/6 checks passed


## 9. 디버그 정보

In [11]:
print("="*70)
print("🔍 디버그 정보")
print("="*70)
print(f"Result keys: {result.keys()}")
print(f"Errors: {result.get('errors', [])}")
print(f"Completed agents: {result.get('completed_agents', [])}")

🔍 디버그 정보
Result keys: dict_keys(['char_relations', 'completed_agents', 'messages'])
Errors: []
Completed agents: ['relations']
